In [2]:
import pandas as pd
import duckdb
from prefect import flow, task

In [3]:
datos = pd.read_excel(r"/workspaces/Proyecto_Aula_2/DATOS PROYECTO (1).xlsx")
datos = datos
datos = datos.astype(str)
datos.columns = [i.replace("/", "").replace("Ó", "O").replace("É", "E").replace(" ","_") for i in datos.columns]

In [3]:
#datos = datos

In [4]:
#datos = datos.astype(str)

In [3]:
datos.info()
datos.describe()

<class 'pandas.DataFrame'>
RangeIndex: 251 entries, 0 to 250
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   FECHA          251 non-null    str  
 1   EURUSD         251 non-null    str  
 2   INTERES_USA    251 non-null    str  
 3   INTERES_EUR    251 non-null    str  
 4   INFLACION_USA  251 non-null    str  
 5   INFLACION_EUR  251 non-null    str  
 6   PIB_USA        251 non-null    str  
 7   PIB_EUR        251 non-null    str  
dtypes: str(8)
memory usage: 15.8 KB


,FECHA,EURUSD,INTERES_USA,INTERES_EUR,INFLACION_USA,INFLACION_EUR,PIB_USA,PIB_EUR
count,251,251,251,251,251,251,251,251
unique,251,244,112,64,249,250,84,84
top,2005-02-01,1.3646,0.09,-0.4,0.0,0.0,12922.656,2293074.8
freq,1,2,20,41,3,2,3,3


In [4]:
[i.replace("/", "").replace("Ó", "O").replace("É", "E").replace(" ","_") for i in datos.columns]

['FECHA',
 'EURUSD',
 'INTERES_USA',
 'INTERES_EUR',
 'INFLACION_USA',
 'INFLACION_EUR',
 'PIB_USA',
 'PIB_EUR']

In [7]:
#datos.info()

In [13]:
@task
def cargar_datos():

    df = pd.read_excel("/workspaces/Proyecto_Aula_2/DATOS PROYECTO (1).xlsx")

    df.columns = [
        'FECHA',
        'EURUSD',
        'INTERES_USA',
        'INTERES_EUR',
        'INFLACION_USA',
        'INFLACION_EUR',
        'PIB_USA',
        'PIB_EUR'
    ]

    return df

In [14]:
@task
def crear_esquemas():

    con = duckdb.connect("proyectolassupernenas.duckdb")

    con.execute("CREATE SCHEMA IF NOT EXISTS proyectolassupernenas.bronze")
    con.execute("CREATE SCHEMA IF NOT EXISTS proyectolassupernenas.silver")
    con.execute("CREATE SCHEMA IF NOT EXISTS proyectolassupernenas.gold")

In [15]:
@task
def bronze(df):

    con = duckdb.connect("proyectolassupernenas.duckdb")

    con.execute("""
    CREATE OR REPLACE TABLE proyectolassupernenas.bronze.precios AS
    SELECT * FROM df
    """)

In [16]:
@task
def silver():

    con = duckdb.connect("proyectolassupernenas.duckdb")

    con.execute("""
    CREATE OR REPLACE TABLE proyectolassupernenas.silver.precios AS
    SELECT
    *,
    CAST(EURUSD AS FLOAT) AS EURUSD_F,
    CAST(INTERES_USA AS FLOAT) AS INTERES_USA_F,
    CAST(INTERES_EUR AS FLOAT) AS INTERES_EUR_F,
    CAST(INFLACION_USA AS FLOAT) AS INFLACION_USA_F,
    CAST(INFLACION_EUR AS FLOAT) AS INFLACION_EUR_F,
    CAST(PIB_USA AS FLOAT) AS PIB_USA_F,
    CAST(PIB_EUR AS FLOAT) AS PIB_EUR_F,
    CAST(FECHA AS DATE) AS FECHA_F
FROM proyectolassupernenas.bronze.precios
""")

In [17]:
@task
def gold():

    con = duckdb.connect("proyectolassupernenas.duckdb")

    con.execute("""
    CREATE OR REPLACE TABLE proyectolassupernenas.gold.modelo AS
    SELECT
        *,
        (INTERES_USA_F - INTERES_EUR_F) AS diff_rate
    FROM proyectolassupernenas.silver.precios
    """)

In [18]:
@task
def analisis():

    con = duckdb.connect("proyectolassupernenas.duckdb")

    df = con.execute("""
    SELECT * FROM proyectolassupernenas.gold.modelo
    """).df()

    print(df.describe())

In [19]:
@flow
def flujo_de_tareas():

    crear_esquemas()

    df = cargar_datos()

    bronze(df)

    silver()

    gold()

    analisis()

In [20]:
flujo_de_tareas()

23:40:33.887 | INFO    | Flow run 'persimmon-quetzal' - Beginning flow run 'persimmon-quetzal' for flow 'flujo-de-tareas'

23:40:33.889 | INFO    | Flow run 'persimmon-quetzal' - View at https://app.prefect.cloud/account/57ae3166-d87a-4de2-8607-dd788bfe6ff8/workspace/c3407c74-a61b-49ca-97c0-3701378b812d/runs/flow-run/06a050bf-1976-7acc-8000-b0647f5c28e2

23:40:33.897 | INFO    | Task run 'crear_esquemas-8f4' - Finished in state Completed()

23:40:33.925 | INFO    | Task run 'cargar_datos-733' - Finished in state Completed()

23:40:33.937 | INFO    | Task run 'bronze-1c6' - Finished in state Completed()

23:40:33.959 | INFO    | Task run 'silver-b57' - Finished in state Completed()

23:40:33.973 | INFO    | Task run 'gold-628' - Finished in state Completed()

                            FECHA      EURUSD  INTERES_USA  INTERES_EUR  \
count                         251  251.000000   251.000000   251.000000   
mean   2015-07-02 01:20:19.123506    1.224317     1.804582     0.781952   
min           2005-02-01 00:00:00    0.985300     0.050000    -0.500000   
25%           2010-04-16 00:00:00    1.113150     0.120000    -0.400000   
50%           2015-07-01 00:00:00    1.196500     0.660000     0.250000   
75%           2020-09-16 00:00:00    1.322950     3.830000     2.000000   
max           2025-12-01 00:00:00    1.575900     5.330000     4.000000   
std                           NaN    0.130529     1.976372     1.415974   

       INFLACION_USA  INFLACION_EUR       PIB_USA       PIB_EUR    EURUSD_F  \
count     251.000000     251.000000    251.000000  2.510000e+02  251.000000   
mean        0.002125       0.001776  19544.470195  2.555594e+06    1.224317   
min        -0.017705      -0.015434  12527.214000  2.273045e+06    0.985300   
25%     

23:40:34.024 | INFO    | Task run 'analisis-a30' - Finished in state Completed()

23:40:34.892 | INFO    | Flow run 'persimmon-quetzal' - Finished in state Completed()

Codigo bien

In [20]:
import duckdb

con = duckdb.connect("proyectolassupernenas.duckdb")

df = con.execute("""
SELECT * 
FROM proyectolassupernenas.gold.modelo
""").df()

df.head()

,FECHA_F,EURUSD_F,diff_rate,diff_inflation,diff_gdp
0,2005-02-01,1.3013,1.50,0.000570,-2260518.0
1,2005-03-01,1.3185,1.63,-0.003787,-2260518.0
2,2005-04-01,1.2943,1.79,-0.001054,-2280152.0
3,2005-05-01,1.2697,2.00,-0.002766,-2280152.0
4,2005-06-01,1.2155,2.04,-0.000547,-2280152.0


In [21]:
df[['EURUSD_F','diff_rate','diff_inflation','diff_gdp']].corr()

,EURUSD_F,diff_rate,diff_inflation,diff_gdp
EURUSD_F,1.000000,-0.482197,0.043930,0.673417
diff_rate,-0.482197,1.000000,0.057003,-0.360734
diff_inflation,0.043930,0.057003,1.000000,-0.015388
diff_gdp,0.673417,-0.360734,-0.015388,1.000000
